# Indices de lexicalité : comparaison des OCR natifs et non natifs

In [ ]:
#import duckdb
#con = duckdb.connect("../data/ocr-fulltext.db");
#con.sql("SELECT id, decode(content) FROM fulltext_ocr_generated LIMIT 100;").df()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp

# ══════════════════════════════════════════════════════════════════════
# DONNÉES  —  remplace par tes deux séries réelles
# ══════════════════════════════════════════════════════════════════════
df_a = pd.read_csv("ocr_lexicality-report-non-natif.csv")
df_b = pd.read_csv("ocr_lexicality-report-natif.csv")
sample_a = df_a["lexicality"].dropna().values
sample_b = df_b["lexicality"].dropna().values
LABEL_A = "Non natif"
LABEL_B = "Natif"

# ══════════════════════════════════════════════════════════════════════
# TESTS STATISTIQUES
# ══════════════════════════════════════════════════════════════════════
u_stat, p_mw  = mannwhitneyu(sample_a, sample_b, alternative="two-sided")
d_stat, p_ks  = ks_2samp(sample_a, sample_b)

# Rank-biserial correlation (effect size de Mann-Whitney)
n_a, n_b = len(sample_a), len(sample_b)
rbc = 1 - (2 * u_stat) / (n_a * n_b)

# Cohen's d
pooled_std = np.sqrt(((n_a - 1) * sample_a.std()**2 + (n_b - 1) * sample_b.std()**2) / (n_a + n_b - 2))
cohens_d   = (sample_a.mean() - sample_b.mean()) / pooled_std

def p_fmt(p):
    if p < 0.001: return "p < 0.001"
    if p < 0.01:  return f"p = {p:.3f}"
    return f"p = {p:.2f}"

def effect_label(d):
    d = abs(d)
    if d < 0.2: return "négligeable"
    if d < 0.5: return "faible"
    if d < 0.8: return "modéré"
    return "fort"

# ══════════════════════════════════════════════════════════════════════
# STYLE
# ══════════════════════════════════════════════════════════════════════
C_A   = plt.cm.viridis(0.2)
C_B   = plt.cm.viridis(0.75)
C_BG  = "white"
C_GRID= "#e8e8e8"
C_TXT = "#1a1a2e"
C_MUT = "#666680"

plt.rcParams.update({
    "figure.facecolor":  C_BG,
    "axes.facecolor":    C_BG,
    "axes.edgecolor":    "#cccccc",
    "axes.labelcolor":   C_TXT,
    "xtick.color":       C_MUT,
    "ytick.color":       C_MUT,
    "text.color":        C_TXT,
    "grid.color":        C_GRID,
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ══════════════════════════════════════════════════════════════════════
# FIGURE  (2 lignes × 3 colonnes)
# ══════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(17, 10))
fig.patch.set_facecolor(C_BG)

gs = gridspec.GridSpec(
    2, 3,
    figure=fig,
    hspace=0.45, wspace=0.35,
    left=0.06, right=0.97, top=0.88, bottom=0.09,
)

# ── Titre principal ───────────────────────────────────────────────────
fig.suptitle(
    f"Comparaison des scores de lexicalité — {LABEL_A} vs {LABEL_B}",
    fontsize=15, weight="bold", y=0.97, color=C_TXT,
)

# ══════════════════════════════════════════════════════════════════════
# 1. KDE (densités superposées)
# ══════════════════════════════════════════════════════════════════════
ax1 = fig.add_subplot(gs[0, :2])   # occupe les 2 premières colonnes

x_kde = np.linspace(0, 100, 500)
kde_a = stats.gaussian_kde(sample_a, bw_method=0.08)
kde_b = stats.gaussian_kde(sample_b, bw_method=0.08)

ax1.fill_between(x_kde, kde_a(x_kde), alpha=0.25, color=C_A)
ax1.fill_between(x_kde, kde_b(x_kde), alpha=0.25, color=C_B)
ax1.plot(x_kde, kde_a(x_kde), linewidth=2.2, color=C_A,
         label=f"{LABEL_A}  (n={n_a:,}, med={np.median(sample_a):.1f} %)")
ax1.plot(x_kde, kde_b(x_kde), linewidth=2.2, color=C_B,
         label=f"{LABEL_B}  (n={n_b:,}, med={np.median(sample_b):.1f} %)")

# Médianes verticales
for samp, color in [(sample_a, C_A), (sample_b, C_B)]:
    ax1.axvline(np.median(samp), color=color, linewidth=1.2,
                linestyle="--", alpha=0.8)

ax1.set_xlabel("Score de lexicalité (%)", fontsize=10)
ax1.set_ylabel("Densité", fontsize=10)
ax1.set_title("Densités de probabilité (KDE)", fontsize=11, weight="semibold")
ax1.legend(fontsize=9, framealpha=0.6)
ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax1.grid(axis="y", alpha=0.5)

# ══════════════════════════════════════════════════════════════════════
# 2. Encart stats
# ══════════════════════════════════════════════════════════════════════
ax_stats = fig.add_subplot(gs[0, 2])
ax_stats.axis("off")

lines = [
    ("Tests statistiques", None, 11, "bold"),
    ("", None, 9, "normal"),
    ("Mann-Whitney U", None, 9, "semibold"),
    (f"  {p_fmt(p_mw)}", None, 9, "normal"),
    (f"  Rank-biserial r = {rbc:.3f}", None, 9, "normal"),
    ("", None, 9, "normal"),
    ("Kolmogorov-Smirnov", None, 9, "semibold"),
    (f"  D = {d_stat:.3f},  {p_fmt(p_ks)}", None, 9, "normal"),
    ("", None, 9, "normal"),
    ("Taille d'effet", None, 9, "semibold"),
    (f"  Cohen's d = {cohens_d:.3f}", None, 9, "normal"),
    (f"  → effet {effect_label(cohens_d)}", None, 9, "italic"),
    ("", None, 9, "normal"),
    ("Statistiques descriptives", None, 9, "semibold"),
    (f"  {LABEL_A}  moy={sample_a.mean():.1f} %  σ={sample_a.std():.1f}", None, 8, "normal"),
    (f"  {LABEL_B}  moy={sample_b.mean():.1f} %  σ={sample_b.std():.1f}", None, 8, "normal"),
]

y = 0.97
for text, _, size, style in lines:
    weight = "bold" if style == "bold" else ("semibold" if style == "semibold" else "normal")
    fontstyle = "italic" if style == "italic" else "normal"
    ax_stats.text(0.04, y, text, transform=ax_stats.transAxes,
                  fontsize=size, fontweight=weight, fontstyle=fontstyle,
                  color=C_TXT, va="top")
    y -= 0.072 if size >= 10 else 0.065

# Cadre de l'encart
for spine in ["top", "bottom", "left", "right"]:
    ax_stats.spines[spine].set_visible(False)
rect = plt.Rectangle((0, 0), 1, 1, transform=ax_stats.transAxes,
                      fill=True, facecolor="#f5f5fa", edgecolor="#ccccdd",
                      linewidth=1, zorder=0)
ax_stats.add_patch(rect)

# ══════════════════════════════════════════════════════════════════════
# 3. Violin plot
# ══════════════════════════════════════════════════════════════════════
ax3 = fig.add_subplot(gs[1, 0])

vp = ax3.violinplot([sample_a, sample_b],
                    positions=[1, 2],
                    showmedians=False, showextrema=False)

for body, color in zip(vp["bodies"], [C_A, C_B]):
    body.set_facecolor(color)
    body.set_alpha(0.55)
    body.set_edgecolor(color)

# Boxplot intérieur
for i, (samp, color) in enumerate([(sample_a, C_A), (sample_b, C_B)], start=1):
    q25, med, q75 = np.percentile(samp, [25, 50, 75])
    ax3.plot([i, i], [q25, q75], color=color, linewidth=5, solid_capstyle="round", alpha=0.6)
    ax3.plot(i, med, "o", color="white", markeredgecolor=color,
             markeredgewidth=1.8, markersize=7, zorder=5)
    ax3.text(i, med + 1.5, f"{med:.1f} %", ha="center", fontsize=8,
             color=color, fontweight="bold")

ax3.set_xticks([1, 2])
ax3.set_xticklabels([LABEL_A, LABEL_B], fontsize=10)
ax3.set_ylabel("Score de lexicalité (%)", fontsize=10)
ax3.set_title("Violin plot", fontsize=11, weight="semibold")
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax3.grid(axis="y", alpha=0.5)
ax3.set_xlim(0.4, 2.6)

# ══════════════════════════════════════════════════════════════════════
# 4. ECDF
# ══════════════════════════════════════════════════════════════════════
ax4 = fig.add_subplot(gs[1, 1])

for samp, color, label in [(sample_a, C_A, LABEL_A), (sample_b, C_B, LABEL_B)]:
    xs = np.sort(samp)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax4.plot(xs, ys * 100, linewidth=2, color=color, label=label)

# Ligne 50 %
ax4.axhline(50, color="#aaaaaa", linewidth=0.8, linestyle=":")
ax4.text(1, 51, "50e centile", fontsize=7.5, color="#aaaaaa")

ax4.set_xlabel("Score de lexicalité (%)", fontsize=10)
ax4.set_ylabel("Centile cumulé (%)", fontsize=10)
ax4.set_title("Fonctions de répartition (ECDF)", fontsize=11, weight="semibold")
ax4.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax4.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax4.legend(fontsize=9, framealpha=0.6)
ax4.grid(alpha=0.4)

# ══════════════════════════════════════════════════════════════════════
# 5. Scatter jitter (dispersion des deux groupes)
# ══════════════════════════════════════════════════════════════════════
ax5 = fig.add_subplot(gs[1, 2])

rng2 = np.random.default_rng(99)
MAX_PTS = 2000

for i, (samp, color, label) in enumerate([(sample_a, C_A, LABEL_A), (sample_b, C_B, LABEL_B)], start=1):
    idx = rng2.choice(len(samp), size=min(MAX_PTS, len(samp)), replace=False)
    sub = samp[idx]
    jitter = rng2.normal(0, 0.12, size=len(sub))
    ax5.scatter(i + jitter, sub, c=sub, cmap="viridis",
                vmin=0, vmax=100, alpha=0.25, s=5, linewidths=0)
    # Médiane
    med = np.median(samp)
    ax5.plot([i - 0.35, i + 0.35], [med, med],
             color=color, linewidth=2.5, solid_capstyle="round", zorder=5)

ax5.set_xticks([1, 2])
ax5.set_xticklabels([LABEL_A, LABEL_B], fontsize=10)
ax5.set_ylabel("Score de lexicalité (%)", fontsize=10)
ax5.set_title(f"Dispersion (échantillon ≤ {MAX_PTS:,} pts)", fontsize=11, weight="semibold")
ax5.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax5.grid(axis="y", alpha=0.4)
ax5.set_xlim(0.4, 2.6)

# ══════════════════════════════════════════════════════════════════════
plt.savefig("fig_comparaison_lexicalite.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : fig_comparaison_lexicalite.png")
